# ✨ sample-data | MCMP Object Storage
위에서부터 셀을 실행하면 **연결된 전체 버킷 → 샘플 파일 → 컬러 표 → KPI·차트**를 볼 수 있습니다.
기본값은 허용된 모든 버킷에서 작은 CSV 파일 하나씩(CSV가 없으면 Parquet)을 선택해 나란히 분석합니다. 파일을 합치지는 않습니다.
파일/컬럼 설정을 바꾸고 아래 셀을 다시 실행하세요. 자동으로 선택한 집계 컬럼이 분석 목적에 맞는지 확인하세요.

CSP 자격증명은 VM에 저장하지 않습니다. AM에서 필요할 때 발급받는 Presigned URL로 파일을 읽습니다.
이 노트북은 버킷을 마운트하지 않으며, 전체 셀을 실행해도 버킷에 업로드하거나 원본을 수정하지 않습니다.
다운로드는 기본 25 MiB로 제한합니다. 압축된 Parquet를 읽을 때 실제 메모리 사용량은 더 커질 수 있습니다.

In [ ]:
import io
import os
from pathlib import Path
import pandas as pd
import requests

gateway = os.environ['MCMP_OBJECT_STORAGE_GATEWAY_URL'].rstrip('/')
access_token = os.environ['MCMP_OBJECT_STORAGE_TOKEN']
auth_headers = {'Authorization': 'Bearer ' + access_token}

def _mcmp(method, path, **kwargs):
    response = requests.request(method, gateway + path, headers=auth_headers, timeout=30, **kwargs)
    response.raise_for_status()
    payload = response.json()
    if payload.get('code') != 200:
        raise RuntimeError(payload.get('detail') or payload.get('message') or 'Application Manager request failed')
    return payload.get('data')

def storages():
    return _mcmp('GET', '/storages')

def list_objects(storage, prefix=''):
    data = _mcmp('GET', '/objects', params={'storage': storage, 'prefix': prefix})
    return pd.DataFrame(data.get('objects', []))

def _presigned(storage, object_key, operation):
    return _mcmp('POST', '/presigned-url', json={'storage': storage, 'objectKey': object_key, 'operation': operation})

def download(storage, object_key, destination=None):
    ticket = _presigned(storage, object_key, 'download')
    response = requests.request(ticket['method'], ticket['presignedURL'], headers=ticket.get('requiredHeaders') or {}, timeout=300)
    response.raise_for_status()
    if destination is None:
        return response.content
    Path(destination).write_bytes(response.content)
    return Path(destination)

def upload(storage, source, object_key):
    ticket = _presigned(storage, object_key, 'upload')
    with Path(source).open('rb') as stream:
        response = requests.request(ticket['method'], ticket['presignedURL'], headers=ticket.get('requiredHeaders') or {}, data=stream, timeout=300)
    response.raise_for_status()
    return object_key

def read_csv(storage, object_key, **kwargs):
    return pd.read_csv(io.BytesIO(download(storage, object_key)), **kwargs)

print('Object Storage helpers are ready.')

In [ ]:
from html import escape
from IPython.display import HTML, display

def _preview_bytes(storage, object_key, limit_bytes):
    """Bound the preview download without printing presigned URLs in errors."""
    ticket = _presigned(storage, object_key, 'download')
    try:
        with requests.request(ticket['method'], ticket['presignedURL'],
                              headers=ticket.get('requiredHeaders') or {},
                              stream=True, timeout=60) as response:
            if not response.ok:
                raise RuntimeError(f"Object download failed (HTTP {response.status_code}).")
            chunks, total = [], 0
            for chunk in response.iter_content(chunk_size=65536):
                total += len(chunk)
                if total > limit_bytes:
                    raise ValueError('File exceeds MAX_DOWNLOAD_MB. Choose a smaller file or raise the limit.')
                chunks.append(chunk)
            return b''.join(chunks)
    except requests.RequestException:
        raise RuntimeError('Object download failed. Check connectivity and retry to obtain a fresh URL.') from None

def _choose_preview_key(items, requested_key, limit_bytes):
    supported = [o for o in items if str(o.get('key', '')).lower().endswith(('.csv', '.parquet'))]
    if requested_key:
        selected = next((o for o in supported if o['key'] == requested_key), None)
        if selected is None:
            raise ValueError('OBJECT_KEY must be a listed CSV or Parquet file within the allowed prefix.')
        if int(selected.get('size') or 0) > limit_bytes:
            raise ValueError('Selected file exceeds MAX_DOWNLOAD_MB.')
        return requested_key
    supported = [o for o in supported if not any(part.startswith('.') for part in o['key'].split('/'))]
    supported.sort(key=lambda o: (not o['key'].lower().endswith('.csv'), o['key']))
    return next((o['key'] for o in supported if int(o.get('size') or 0) <= limit_bytes), None)

def _summarize(data, group_col=None, value_col=None, aggregation='sum'):
    if data is None or data.empty:
        return None, group_col, value_col
    if aggregation not in {'sum', 'mean', 'min', 'max', 'count'}:
        raise ValueError('AGGREGATION must be sum, mean, min, max or count.')
    text_cols = list(data.select_dtypes(include=['object', 'string', 'category']).columns)
    num_cols = list(data.select_dtypes(include='number').columns)
    group_col = group_col or ('region' if 'region' in data.columns else (text_cols[0] if text_cols else None))
    value_col = value_col or (num_cols[0] if num_cols else None)
    if group_col is None or value_col is None:
        return None, group_col, value_col
    if group_col not in data.columns or value_col not in data.columns:
        raise ValueError('GROUP_COL and VALUE_COL must name columns in the loaded table.')
    if value_col not in num_cols:
        raise ValueError('VALUE_COL must be numeric. Check the CSV format or choose another column.')
    clean = pd.DataFrame({
        'group_key': data[group_col].astype('string').fillna('(missing)'),
        'value': pd.to_numeric(data[value_col], errors='coerce')
    })
    clean = clean[clean['value'].notna() & ~clean['value'].isin([float('inf'), float('-inf')])]
    if clean.empty:
        return None, group_col, value_col
    summary = clean.groupby('group_key', dropna=False)['value'].agg(aggregation).reset_index()
    return summary.sort_values('value', ascending=False), group_col, value_col

## 1. 연결된 Object Storage
배포 시 선택한 모든 버킷과 접근 권한을 표시합니다. 다른 CSP의 버킷도 각각 별도 카드와 결과로 나타납니다.

In [ ]:
available_storages = storages() or []
if not available_storages:
    print('No granted storage found. Check the Object Storage selection in Application Manager.')
else:
    storage_frame = pd.DataFrame(available_storages)
    display(storage_frame.style.set_caption('Granted Object Storage').set_table_styles([{'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', '700'), ('color', '#1e3a8a'), ('text-align', 'left')]}, {'selector': 'th', 'props': [('background-color', '#eff6ff'), ('color', '#1e3a8a')]}]).set_properties(**{'border-color': '#dbeafe'}))

## 2. 버킷·파일 선택
`STORAGE_ALIAS = None`이면 연결된 **모든 스토리지**를 분석합니다. alias를 지정하면 해당 스토리지만 분석합니다.
`OBJECT_PREFIX`와 `OBJECT_KEY`는 공통 기본값이며, 스토리지별 설정은 `PREFIX_BY_ALIAS`와 `OBJECT_KEY_BY_ALIAS`에 지정할 수 있습니다.
예: `PREFIX_BY_ALIAS = {'aws-data': 'sample-data/', 'ncp-data': 'reports/'}`.
설정을 변경하면 **3번부터 다시 실행**하세요.

In [ ]:
STORAGE_ALIAS = None  # None: all connected storages
OBJECT_PREFIX = ''    # Common fallback prefix
OBJECT_KEY = None     # Common fallback object key
PREFIX_BY_ALIAS = {}  # Example: {'aws-data': 'sample-data/', 'ncp-data': 'reports/'}
OBJECT_KEY_BY_ALIAS = {}  # Example: {'aws-data': 'sample-data/csv/usage.csv'}
MAX_DOWNLOAD_MB = 25
PREVIEW_ROWS = 10

## 3. 버킷 파일 목록
목록은 화면에 최대 50개를 표시합니다. 버킷에 새 파일을 올리면 이 셀부터 다시 실행하세요.

In [ ]:
df, summary, selected_key, selected_storage = None, None, None, None
objects = pd.DataFrame()
selected_storages, storage_objects = [], {}
if not available_storages:
    print('Run section 1 and check the granted storages.')
else:
    if STORAGE_ALIAS is None:
        selected_storages = list(available_storages)
    else:
        selected = next((s for s in available_storages if s['alias'] == STORAGE_ALIAS), None)
        if selected is None:
            raise ValueError('STORAGE_ALIAS must match an alias shown in section 1.')
        selected_storages = [selected]
    for storage in selected_storages:
        alias = storage['alias']
        prefix = PREFIX_BY_ALIAS.get(alias, OBJECT_PREFIX)
        listed = list_objects(alias, prefix)
        storage_objects[alias] = listed
        provider = escape(str(storage.get('provider') or storage.get('providerName') or 'Object Storage'))
        display(HTML(f'''<div style="display:flex;justify-content:space-between;align-items:center;padding:10px 14px;margin:16px 0 8px;border-left:5px solid #3b82f6;background:#f8fafc;border-radius:10px"><b style="color:#0f172a">{escape(str(alias))}</b><span style="color:#475569">{provider} · {len(listed)} objects</span></div>'''))
        display(listed.head(50))
        if listed.empty:
            print(f'No objects found in {alias}. Check its prefix or upload data, then rerun this cell.')
    if selected_storages:
        selected_storage = selected_storages[0]
        objects = storage_objects[selected_storage['alias']]

## 4. 데이터 미리보기
각 Object Storage에서 CSV 또는 Parquet 파일 **하나씩** 읽습니다. 숨김 파일·폴더는 자동 선택하지 않습니다.
다운로드 크기를 넘는 파일은 자동 선택에서 제외합니다. 큰 데이터는 별도의 분석 작업으로 처리하세요.
Parquet에는 pyarrow 또는 fastparquet가 필요합니다. CSV 옵션이 필요하면 아래 `CSV_OPTIONS`를 수정하세요.

In [ ]:
df, summary, selected_key = None, None, None
loaded_data, selected_keys, load_errors = {}, {}, {}
CSV_OPTIONS = {}  # Example: {'encoding': 'utf-8', 'sep': ','}
if MAX_DOWNLOAD_MB <= 0 or PREVIEW_ROWS <= 0:
    raise ValueError('MAX_DOWNLOAD_MB and PREVIEW_ROWS must be positive.')
limit_bytes = int(MAX_DOWNLOAD_MB * 1024 * 1024)
for storage in selected_storages:
    alias = storage['alias']
    listed = storage_objects.get(alias, pd.DataFrame())
    if listed.empty:
        continue
    requested_key = OBJECT_KEY_BY_ALIAS.get(alias, OBJECT_KEY)
    try:
        key = _choose_preview_key(listed.to_dict('records'), requested_key, limit_bytes)
        if key is None:
            raise ValueError('No CSV/Parquet file within the download limit.')
        payload = _preview_bytes(alias, key, limit_bytes)
        if key.lower().endswith('.parquet'):
            current_df = pd.read_parquet(io.BytesIO(payload))
        else:
            current_df = pd.read_csv(io.BytesIO(payload), **CSV_OPTIONS)
    except ImportError:
        load_errors[alias] = 'Parquet support is missing. Use CSV or install pyarrow/fastparquet.'
        print(f'{alias}: {load_errors[alias]}')
        continue
    except pd.errors.EmptyDataError:
        load_errors[alias] = 'The selected CSV contains no data.'
        print(f'{alias}: {load_errors[alias]}')
        continue
    except (RuntimeError, ValueError, requests.RequestException) as error:
        load_errors[alias] = str(error)
        print(f'{alias}: {load_errors[alias]}')
        continue
    finally:
        if 'payload' in locals():
            del payload
    loaded_data[alias], selected_keys[alias] = current_df, key
    preview = current_df.head(PREVIEW_ROWS)
    preview_style = preview.style.set_caption(f'{alias} · sample rows').set_table_styles([{'selector': 'caption', 'props': [('font-size', '15px'), ('font-weight', '700'), ('color', '#1e3a8a'), ('text-align', 'left')]}, {'selector': 'th', 'props': [('background-color', '#eff6ff'), ('color', '#1e3a8a')]}]).format(precision=2, na_rep='—')
    preview_numeric = list(preview.select_dtypes(include='number').columns)
    if preview_numeric:
        preview_style = preview_style.background_gradient(cmap='Blues', subset=preview_numeric)
    display(preview_style)
if loaded_data:
    first_alias = next(iter(loaded_data))
    df, selected_key = loaded_data[first_alias], selected_keys[first_alias]
    selected_storage = next(s for s in selected_storages if s['alias'] == first_alias)
    objects = storage_objects[first_alias]
else:
    print('No sample data was loaded. Check each storage result above.')

## 5. 집계와 차트
그룹 컬럼은 기본적으로 `region` 또는 첫 문자열 컬럼, 값 컬럼은 첫 숫자 컬럼입니다.
**자동 선택된 컬럼과 단위를 반드시 확인**하고 원하는 `GROUP_COL`, `VALUE_COL`을 지정하세요.
`AGGREGATION`: sum / mean / min / max / count. count는 선택한 값 컬럼의 유효한 값 개수입니다.
각 스토리지에서 선택한 파일을 별도 대시보드로 표시합니다. 차트는 집계값 상위 `TOP_N`개와 원본 값의 분포를 함께 보여줍니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

GROUP_COL = None
VALUE_COL = None
AGGREGATION = 'sum'
TOP_N = 20

summary, summaries = None, {}
if TOP_N <= 0:
    raise ValueError('TOP_N must be positive.')
if not loaded_data:
    print('No loaded data. Run section 4 first.')
else:
    for alias, current_df in loaded_data.items():
        current_summary, group_col, value_col = _summarize(current_df, GROUP_COL, VALUE_COL, AGGREGATION)
        if current_summary is None:
            print(f'{alias}: no suitable grouping/numeric columns. Set GROUP_COL and VALUE_COL explicitly.')
            continue
        summaries[alias] = current_summary
        styled_summary = current_summary.head(TOP_N).style.set_caption(f'{alias} · {AGGREGATION}({value_col}) by {group_col}').background_gradient(cmap='Blues', subset=['value']).format({'value': '{:,.2f}'})
        display(styled_summary)
        chart_data = current_summary.head(TOP_N).sort_values('value')
        raw_values = pd.to_numeric(current_df[value_col], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
        fig, (bar_ax, hist_ax) = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={'width_ratios': [1.7, 1]}, facecolor='#f8fafc')
        colors = plt.cm.Blues(np.linspace(0.45, 0.9, len(chart_data)))
        bars = bar_ax.barh(chart_data['group_key'].astype(str), chart_data['value'], color=colors, edgecolor='white', linewidth=0.8)
        value_span = max(float(chart_data['value'].max() - chart_data['value'].min()), float(chart_data['value'].abs().max()), 1.0)
        for bar, value in zip(bars, chart_data['value']):
            offset = value_span * 0.02
            bar_ax.text(value + (offset if value >= 0 else -offset), bar.get_y() + bar.get_height() / 2, f'{value:,.2f}', va='center', ha='left' if value >= 0 else 'right', fontsize=9, color='#334155')
        bar_ax.set_title(f'{alias} · Top {min(TOP_N, len(current_summary))} groups', loc='left', fontsize=15, fontweight='bold', color='#0f172a')
        bar_ax.set_xlabel(f'{AGGREGATION}({value_col})')
        bar_ax.set_ylabel(str(group_col))
        bins = min(18, max(5, int(np.sqrt(max(len(raw_values), 1)))))
        hist_ax.hist(raw_values, bins=bins, color='#60a5fa', alpha=0.88, edgecolor='white')
        if not raw_values.empty:
            hist_ax.axvline(raw_values.median(), color='#1d4ed8', linestyle='--', linewidth=2, label=f'median {raw_values.median():,.2f}')
            hist_ax.legend(frameon=False)
        hist_ax.set_title('Value distribution', loc='left', fontsize=15, fontweight='bold', color='#0f172a')
        hist_ax.set_xlabel(str(value_col))
        hist_ax.set_ylabel('Rows')
        for ax in (bar_ax, hist_ax):
            ax.set_facecolor('white')
            ax.grid(axis='x', color='#e2e8f0', linewidth=0.8)
            for spine in ('top', 'right', 'left'):
                ax.spines[spine].set_visible(False)
        fig.suptitle(f'{alias} · {selected_keys.get(alias, "sample data")}', x=0.06, ha='left', fontsize=18, fontweight='bold', color='#0f172a')
        plt.tight_layout()
        plt.show()
        plt.close(fig)
    if summaries:
        summary = next(iter(summaries.values()))

## 6. 결과 저장 (선택)
기본 실행은 읽기 전용입니다. 아래 예제는 **주석을 해제할 때만** 실행됩니다.
로컬 CSV 저장은 VM 작업 폴더에 저장합니다. 버킷 업로드는 READ_WRITE 권한과 허용된 경로가 필요합니다.
원본을 덮어쓰지 않도록 결과 파일 경로를 확인하세요.

In [ ]:
# for alias, result in summaries.items():
#     local_file = f'{alias}-summary.csv'
#     result.to_csv(local_file, index=False)  # Local VM file only.
#     upload(alias, local_file, f'results/{local_file}')